# Self-Verification of the Episode-Level Drafts

Runs the 5 checks against `arrhythmia_episodes_draft.csv` and `arrhythmia_features_updated_draft.csv` (built in `extract_arrhythmia_episodes.ipynb`), prints every result explicitly, and only then saves the final `arrhythmia_episodes_482.csv` and overwrites `arrhythmia_features_482.csv`.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE_DIR = Path("..").resolve()
DATA_INTERIM = BASE_DIR / "data" / "interim"

EPISODES_DRAFT_CSV = DATA_INTERIM / "arrhythmia_episodes_draft.csv"
FEATURES_UPDATED_DRAFT_CSV = DATA_INTERIM / "arrhythmia_features_updated_draft.csv"

EPISODES_FINAL_CSV = DATA_INTERIM / "arrhythmia_episodes_482.csv"
FEATURES_FINAL_CSV = DATA_INTERIM / "arrhythmia_features_482.csv"

CLINICAL_CSV = DATA_INTERIM / "imputed_477_cases.csv"

episodes = pd.read_csv(EPISODES_DRAFT_CSV)
features = pd.read_csv(FEATURES_UPDATED_DRAFT_CSV)
clinical = pd.read_csv(CLINICAL_CSV)
N_PATIENTS = len(clinical)

print(f"episodes draft shape: {episodes.shape}")
print(f"features draft shape: {features.shape}")
print(f"Number of patients in cohort: {N_PATIENTS}")

## Check 1: Shape and Episode Ordering

In [ ]:
print(f"episodes shape: {episodes.shape}")
print(f"More rows than {N_PATIENTS} patients: {len(episodes) > N_PATIENTS}")

bad_order = episodes[episodes["episode_end_sec"] < episodes["episode_start_sec"]]
print(f"Rows where episode_end_sec < episode_start_sec: {len(bad_order)}")
if len(bad_order) > 0:
    print(bad_order)

## Check 2: Per-Patient Duration Sum Matches total_arrhythmia_burden_sec

In [ ]:
computed_sum = episodes.groupby("caseid")["episode_duration_sec"].sum().rename("computed_sum")
check2 = features[["caseid", "total_arrhythmia_burden_sec"]].merge(computed_sum, on="caseid", how="left")
check2["computed_sum"] = check2["computed_sum"].fillna(0.0)   # 0-episode patients sum to 0

mismatches = check2[(check2["total_arrhythmia_burden_sec"] - check2["computed_sum"]).abs() > 0.01]
print(f"Mismatches (>0.01s difference): {len(mismatches)}")
if len(mismatches) > 0:
    print(mismatches)

## Check 3: first_episode_start_sec Equals min(episode_start_sec) Per Patient

In [ ]:
computed_min = episodes.groupby("caseid")["episode_start_sec"].min().rename("computed_min")
has_episodes = features[features["total_episode_count"] > 0]
check3 = has_episodes[["caseid", "first_episode_start_sec"]].merge(computed_min, on="caseid", how="left")

mismatches3 = check3[(check3["first_episode_start_sec"] - check3["computed_min"]).abs() > 1e-9]
print(f"Patients checked (those with >0 episodes): {len(check3)}")
print(f"Mismatches: {len(mismatches3)}")
if len(mismatches3) > 0:
    print(mismatches3)

zero_episode_patients = features[features["total_episode_count"] == 0]
print(f"\nPatients with 0 episodes (first_episode_start_sec correctly NaN, not checked above): "
      f"{zero_episode_patients['caseid'].tolist()}")

## Check 4: Episode Count Distribution (Independently Recomputed From the Draft)

In [ ]:
episode_counts_per_patient = episodes.groupby("caseid").size().reindex(clinical["caseid"], fill_value=0)

bins = pd.cut(episode_counts_per_patient, bins=[-0.1, 0, 1, 5, np.inf], labels=["0", "1", "2-5", "5+"])
print("Episode count distribution:")
print(bins.value_counts().sort_index())
print(f"\nTotal valid episodes across all patients: {len(episodes)}")

## Check 5: 3 Multi-Episode Patients — Confirm Gaps All Exceed 30 Seconds

In [ ]:
EPISODE_GAP_SEC = 30.0  # must match the threshold used during extraction

multi_episode_caseids = (
    episodes.groupby("caseid").size()[lambda s: s > 1].index.tolist()[:3]
)
print(f"Sample patients with >1 episode: {multi_episode_caseids}")

for cid in multi_episode_caseids:
    rows = episodes[episodes["caseid"] == cid].sort_values("episode_number")
    print(f"\n--- caseid {cid} ---")
    print(rows.to_string())

    ends = rows["episode_end_sec"].values[:-1]
    next_starts = rows["episode_start_sec"].values[1:]
    gaps = next_starts - ends
    print(f"Gaps between consecutive episodes: {gaps}")
    print(f"All gaps > {EPISODE_GAP_SEC}s: {(gaps > EPISODE_GAP_SEC).all()}")

## Save Final Outputs

In [ ]:
assert episodes["episode_end_sec"].ge(episodes["episode_start_sec"]).all()
assert len(mismatches) == 0
assert len(mismatches3) == 0
assert features["caseid"].is_unique

episodes.to_csv(EPISODES_FINAL_CSV, index=False)
features.to_csv(FEATURES_FINAL_CSV, index=False)
print(f"Saved final episodes table: {EPISODES_FINAL_CSV.resolve()} ({episodes.shape[0]} rows x {episodes.shape[1]} cols)")
print(f"Saved final updated features table: {FEATURES_FINAL_CSV.resolve()} ({features.shape[0]} rows x {features.shape[1]} cols)")